# Huấn luyện mô hình bóc tách CV (Resume Parser) bằng Unsloth & QLoRA

Notebook này hướng dẫn bạn fine-tune các dòng mô hình (ví dụ: `Qwen2.5-3B-Instruct` hoặc `Qwen2.5-7B-Instruct`) bằng thư viện **Unsloth** (nhanh hơn gấp 2-5 lần và tiết kiệm RAM GPU hơn so với HuggingFace truyền thống).

Mô hình sau khi huấn luyện xong sẽ có khả năng trích xuất văn bản CV thô thành cấu trúc JSON chuẩn theo schema mẫu của dự án.

### 1. Cài đặt các thư viện cần thiết
Chạy ô lệnh dưới đây để cài đặt Unsloth và các thư viện dependencies.

In [ ]:
# Cài đặt Unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Cài đặt các thư viện bổ trợ cho trl/peft
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

### 2. Nạp Mô hình & Tokenizer bằng Unsloth
Chúng ta sẽ nạp base model ở định dạng quantized 4-bit để tiết kiệm VRAM (chạy mượt trên GPU T4 của Colab miễn phí).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # Độ dài ngữ cảnh tối đa (hỗ trợ đọc CV dài)
dtype = None # Tự động dò tìm (Float16 cho T4/V100, Bfloat16 cho A100)
load_in_4bit = True # Bật chế độ 4-bit để chạy trên GPU RAM thấp

# Chúng ta chọn dòng Qwen2.5-3B-Instruct hoặc Qwen2.5-7B-Instruct (rất mạnh về xử lý tiếng Việt/Anh và JSON)
model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 3. Cấu hình Parameter-Efficient Fine-Tuning (LoRA / QLoRA)
Chỉ cập nhật 1-2% trọng số của mô hình thông qua adapter để tăng tốc độ huấn luyện.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank của LoRA (khuyên dùng 16 hoặc 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Tối ưu hóa cho Unsloth (bắt buộc phải là 0)
    bias = "none",    # Tối ưu hóa (bắt buộc phải là "none")
    use_gradient_checkpointing = "unsloth", # Tiết kiệm VRAM lên đến 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 4. Chuẩn bị dữ liệu huấn luyện
Hãy upload file `train_dataset_sharegpt.json` từ máy của bạn lên Colab (kéo thả vào thanh công cụ Files ở bên trái). Ô lệnh dưới đây sẽ đọc dữ liệu và đưa vào định dạng hội thoại chuẩn của Qwen (ChatML).

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Áp dụng template hội thoại của Qwen vào mô hình
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT mapping
)

def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = []
    for con in conversations:
        # Định dạng hội thoại chuẩn
        text = tokenizer.apply_chat_template(con, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts, }

# Nạp tệp JSON đã chuẩn bị từ trước
dataset = load_dataset("json", data_files="train_dataset_sharegpt.json", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

### 5. Cấu hình & Tiến hành Huấn luyện (SFT Training)
Sử dụng thư viện TRL `SFTTrainer` để tiến hành huấn luyện.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Cho phép padding động để giảm thiểu thời gian huấn luyện đối với các CV ngắn
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # Có thể tăng lên 200 - 300 nếu cần độ hội tụ sâu hơn
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Bắt đầu train!
trainer_stats = trainer.train()

### 6. Kiểm thử nhanh (Inference Test)
Chúng ta sẽ chuyển mô hình sang trạng thái Inference để kiểm tra kết quả.

In [ ]:
FastLanguageModel.for_inference(model)

sample_cv = """
Tammy Jones
XYZ Accountants, Sacramento, CA January 2012 - Present
Accounting Specialist
- Ensure accuracy of all transactions with clients, vendors, and employees
- Identified and averted over $250,000 in lost revenue from bank errors
"""

instruction = "Hãy đọc đoạn text CV (OCR) dưới đây và trích xuất thông tin thành đúng định dạng JSON theo schema mẫu của cv-template."
messages = [
    {"role": "user", "content": f"{instruction}\n\nTEXT CV:\n{sample_cv}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Bắt buộc phải là True cho inference
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 1024, use_cache = True)
decoded_output = tokenizer.batch_decode(outputs)

print("=== KẾT QUẢ TRÍCH XUẤT THỬ NGHIỆM ===")
print(decoded_output[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", ""))

### 7. Xuất & Lưu Trữ Model
Chúng ta sẽ lưu adapter để tải về máy và cấu hình nén mô hình sang định dạng GGUF phục vụ chạy offline.

In [ ]:
# 7.1 Lưu LoRA adapters (Chỉ nặng ~100MB)
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")
print("Đã lưu lora_adapters thành công!")

# 7.2 Lưu và nén sang định dạng GGUF (kiểu lượng hóa Q4_K_M cực tốt cho CPU/GPU PC cá nhân)
# Lưu ý: Quá trình này sẽ mất khoảng 5-10 phút để export và compile.
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method = "q4_k_m")
print("Đã xuất file GGUF lượng hóa Q4_K_M thành công vào folder 'gguf_model'!")

Bây giờ, bạn chỉ cần nén 2 thư mục `lora_adapters` và `gguf_model` tải về máy và đặt vào thư mục `exports/` trong dự án `cv-llm-finetune-project` là hoàn thành!